# RAG-Powered ICU Nutrition Assistant


## Project Overview

In this project, we build a complete **Retrieval-Augmented Generation (RAG) system** trained on ASPEN's 2016 and 2021 ICU Nutrition guidlines. This tool is intended to assist dietetic interns and new trainees in answering questions on establised best practice in the ICU.

## Part 1: Setup and Configuration

Installing and importing the required libraries.

In [2]:
# Import all required libraries
import pymupdf4llm
import chromadb
from chromadb.utils import embedding_functions
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import torch
import re
from typing import List, Dict, Optional
import json

import os

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("\n All imports successful!")

PyTorch version: 2.14.1+cu130
CUDA available: True

 All imports successful!


---

## Part 2: Project Declaration


In [3]:
# ============================================================
# TODO: Fill in your project details
# ============================================================

PROJECT_INFO = {
    "scenario": "Personal Interest",
    "topic": "Nutrition Intervention in the ICU",
    "description": "This RAG LLM is trained with ASPENs 2016 and 2021 practice guidelines for nutrition intervention in the ICU.",
    "target_users": "RDs and Dietetic Interns",
}

# Print your declaration
print(" PROJECT DECLARATION")
print("=" * 50)
for key, value in PROJECT_INFO.items():
    print(f"{key.replace('_', ' ').title()}: {value}")

 PROJECT DECLARATION
Scenario: Personal Interest
Topic: Nutrition Intervention in the ICU
Description: This RAG LLM is trained with ASPENs 2016 and 2021 practice guidelines for nutrition intervention in the ICU.
Target Users: RDs and Dietetic Interns


---

## Part 3: Creatinig the Knowledge Base

This is the foundation of your RAG system. Create comprehensive, well-organized documents for your chosen scenario.

## Helper Functions

In [4]:
import os
import zipfile

from pathlib import Path

import requests

# Setup path to data folder
data_path = Path("paper_markdowns/")

# If the image folder doesn't exist, download it and prepare it...
if data_path.is_dir():
    print(f"{data_path} directory exists.")
else:
    print(f"Did not find {data_path} directory, creating one...")
    data_path.mkdir(parents=True, exist_ok=True)

import requests

with open('paper_markdowns/ASPEN_2016_guidelines.md', 'wb') as f:
    url = 'https://raw.githubusercontent.com/raphi-l/RAG-ICU-nutrition-assistant-/refs/heads/main/paper-markdowns/ASPEN_2016_ICU_Guidelines.md'
    response = requests.get(url)
    f.write(response.content)

Did not find paper_markdowns directory, creating one...


In [5]:
CACHE_DIR = '/content/paper_markdowns/ASPEN_2016_guidelines.md'

def pdf_to_markdown(path: str) -> str:
  """
  Renders PDF as a markdown file and stores in globally defined `CACHE_DIR`
  """
  filename = os.path.splitext(os.path.basename(path))[0]
  cache_path = os.path.join(CACHE_DIR, f"{filename}.md")

  print(f"[INFO] Parsing '{filename}.pdf'...")

  markdown = pymupdf4llm.to_markdown(path)
  with open(cache_path, "w", encoding="utf-8") as f:
      f.write(markdown)
  print(f"[INFO] PDF saved as markdown to '{cache_path}'")

  return None

In [6]:
def markdown_to_dict(markdown: str,
                     title: str = "") -> dict:
    """Convert markdown into a dictionary of {section_header: content}"""
    sections = {}

    # find all header positions
    matches = list(SECTION_PATTERN.finditer(markdown))

    for i, match in enumerate(matches):
        header = match.group().strip()
        start = match.end()
        # clean up markdown formatting from header for use as key
        key = re.sub(r'[#*_]', '', header).strip()

        # content goes from end of this header to start of next header
        end = matches[i + 1].start() if i + 1 < len(matches) else len(markdown)
        content = f'[{title}][{key}] {markdown[start:end].strip()}'


        sections[key] = content

    return sections

In [7]:
import re

SECTION_PATTERN = re.compile(
    r'^#{2,3}\s+'
    r'(?:'
    r'\*\*[A-Z]\.\s+.+|'              # **A. through **R.
    r'_Trauma_|'
    r'_Traumatic.+|'
    r'_Open\s+Abdomen_|'
    r'_Burns_'
    r')',                              # Introduction removed
    re.IGNORECASE | re.MULTILINE
)

# usage
with open("/content/paper_markdowns/ASPEN_2016_guidelines.md", "r") as f:
    markdown = f.read()

knowledge_base = markdown_to_dict(markdown, title='ASPEN_2016_guidelines')

In [8]:
# find all matches
matches = SECTION_PATTERN.findall(markdown)

# print them
for match in matches:
    print(match)

## **A. Nutrition Assessment** 
## **B. Initiate EN** 
## **C. Dosing of EN** 
## **D. Monitoring Tolerance and Adequacy of EN** 
## **E. Selection of Appropriate Enteral Formulation** 
## **F. Adjunctive Therapy** 
## **G. When to Use PN** 
## **H. When Indicated, Maximize Efficacy of PN** 
## **I. Pulmonary Failure** 
## **J. Renal Failure** 
## **K. Hepatic Failure** 
## **L. Acute Pancreatitis** 
## **M. Surgical Subsets** 
## _Trauma_
## _Traumatic Brain Injury_ 
## _Open Abdomen_
## _Burns_
## **N. Sepsis** 
## **O. Postoperative Major Surgery (SICU Admission Expected)** 
## **P. Chronically Critically Ill** 
## **Q. Obesity in Critical Illness** 
## **R. Nutrition Therapy End-of-Life Situations** 


In [9]:
def clean_section(text: str) -> str:

    patterns = [
        # ── 1. fix italic P values
        (r'_P_', 'P'),

        # ── 2. Question/Rationale labels
        (r'_\*\*Rationale:\*\*_', 'Rationale:'),
        (r'_\*\*Rationale:\*\*', 'Rationale:'),
        (r'\*\*Rationale:\*\*_', 'Rationale:'),
        (r'\*\*Rationale:\*\*', 'Rationale:'),
        (r'_Rationale:_', 'Rationale:'),
        (r'Rationale:_', 'Rationale:'),
        (r'_Question:_', 'Question:'),
        (r'Question:_', 'Question:'),
        (r'_Question:', 'Question:'),

        # ── 2.5 preserve mid-recommendation continuations before anything removes them
        (r'##\s+\*\*(intake\b[^\n]*)\*\*', r'\1'),
        (r'##\s+\*\*(the critically\b[^\n]*)\*\*', r'\1'),
        (r'##\s+\*\*(specialty\b[^\n]*)\*\*', r'\1'),
        (r'##\s+\*\*(in the absence\b[^\n]*)\*\*', r'\1'),

        # ── 3. fix mid-sentence page breaks
        (r'(=|\(|\)|,|\w|\*)\s*\n\n_McClave et al_\s*\n\n\d+\s*\n\n\*\*==>.*?<==\*\*\n\n\*\*Figure\s+\d+\.\*\*[^\n]*\n\n', r'\1 '),
        (r'(=|\(|\)|,|\w|\*)\s*\n\n_McClave et al_\s*\n\n\d+\s*\n\n(?:\*\*==>.*?<==\*\*\n\n\*\*Figure\s+\d+\.\*\*[^\n]*\n\n)+', r'\1 '),
        (r'(=|\(|\)|,|\w|\*)\s*\n\n_McClave et al_\s*\n\n\d+\s*\n\n\*\*Table\s+\d+\.\*\*[^\n]*\n', r'\1 '),
        (r'(=|\(|\)|,|\w|\*)\s*\n\n_McClave et al_\s*\n\n\d+\s*\n\n(?=\*\*[a-z])', r'\1 '),
        (r'(=|\(|\)|,|\w|\*)\s*\n\n_Journal of Parenteral and Enteral Nutrition[^\n]*_\s*\n\n\d+\s*\n\n\*\*==>.*?<==\*\*\n\n\*\*Figure\s+\d+\.\*\*[^\n]*\n\n', r'\1 '),
        (r'(=|\(|\)|,|\w|\*)\s*\n\n_Journal of Parenteral and Enteral Nutrition[^\n]*_\s*\n\n\d+\s*\n\n(?:\*\*==>.*?<==\*\*\n\n\*\*Figure\s+\d+\.\*\*[^\n]*\n\n)+', r'\1 '),
        (r'(=|\(|\)|,|\w|\*)\s*\n\n_Journal of Parenteral and Enteral Nutrition[^\n]*_\s*\n\n\d+\s*\n\n\*\*Table\s+\d+\.\*\*[^\n]*\n', r'\1 '),
        (r'(=|\(|\)|,|\w|\*)\s*\n\n_Journal of Parenteral and Enteral Nutrition[^\n]*_\s*\n\n\d+\s*\n\n(?=\*\*[a-z])', r'\1 '),
        (r'\n\n\d{3}\s*\n\n(?:\*\*==>.*?<==\*\*\n\n\*\*Figure\s+\d+\.\*\*[^\n]*\n\n)+', ' '),

        # ── 4. remove figures and captions
        (r'\*\*==>.*?<==\*\*\n\n\*\*Figure\s+\d+\.\*\*[^\n]*\n', ''),
        (r'\*\*==>.*?<==\*\*', ''),
        (r'==>.*?<==', ''),
        (r'\n\*\*Figure\s+\d+\.\*\*[^\n]*\n', '\n'),

        # ── 5. remove table captions then table rows
        (r'\*\*Table\s+\d+\.\*\*[^\n]*\n', ''),
        (r'(?:\|.*\n?)+', ''),

        # ── 6. remove bundle statements
        (r'-\s*[●\x07]\s*.*\n', ''),
        (r'-\s*●\s*.*\n', ''),

        # ── 7. remove journal metadata and page numbers
        (r'_Journal of Parenteral and Enteral Nutrition[^\n]*_\s*\n', ''),
        (r'\n\n_McClave et al_\s*\n\n\d+\s*\n\n', ' '),
        (r'_McClave et al_\s*\n', ''),
        (r'^\d{3}\s*$\n?', ''),
        (r'(?<=\w)\s*\n\n\d{3}\s*\n\n(?=\S)', ' '),
        (r'GRADE,.*?\n', ''),
        (r'Adapted from.*?\n', ''),

        # ── 8. remove stray ## headers
        (r'##\s+\*\*\[.*?\]\*\*\s*\n', ''),
        (r'##\s+\*\*[A-Z]\d+[a-z]?\..*?\n', ''),
        (r'##\s+\*\*H4\..*?\n', ''),
        (r'##\s+\*\*K2\..*?\n', ''),
        (r'##\s+\*\*L\d+[a-z]?\..*?\n', ''),
        (r'##\s+\*\*Q\d+\..*?\n', ''),
        (r'##\s+\*\*L3a\..*?\n', ''),

        # ── 9. remove empty headers
        (r'\n\n##\s*\n\n', '\n\n'),
        (r'##\s*\n', ''),
        (r'^\s*##\s*$', ''),

        # ── 10. remove citation numbers
        (r'\[\d+(?:[,;–\-]\d+)*\]', ''),

        # ── 11. strip markdown formatting
        (r'\*{1,2}', ''),
        (r'(?<!\w)_(?=\w)', ''),
        (r'(?<=\w)_(?!\w)', ''),

        # ── 12. whitespace cleanup
        (r'\n{3,}', '\n\n'),
        (r'^\s+$', ''),
    ]

    for pattern, replacement in patterns:
        if r'\|' in pattern:
            text = re.sub(pattern, replacement, text, flags=re.MULTILINE | re.DOTALL)
        else:
            text = re.sub(pattern, replacement, text, flags=re.MULTILINE)

    return text.strip()

In [10]:
knowledge_base['A. Nutrition Assessment']


'[ASPEN_2016_guidelines][A. Nutrition Assessment] _Question:_ Does the use of a nutrition risk indicator identify patients who will most likely benefit from nutrition therapy? \n\n**A1. Based on expert consensus, we suggest a determination of nutrition risk (eg, nutritional risk screening [NRS 2002], NUTRIC score) be performed on all patients admitted to the ICU for whom volitional** \n\n_Journal of Parenteral and Enteral Nutrition 40(2)_ \n\n162 \n\n## **Table 2.** Bundle Statements. \n\n- \x07Assess patients on admission to the intensive care unit (ICU) for nutrition risk, and calculate both energy and protein requirements to determine goals of nutrition therapy. \n\n- ● \x07Initiate enteral nutrition (EN) within 24−48 hours following the onset of critical illness and admission to the ICU, and increase to goals over the first week of ICU stay. \n\n- ● \x07Take steps as needed to reduce risk of aspiration or improve tolerance to gastric feeding (use prokinetic agent, continuous infusi

In [11]:
for key, content in knowledge_base.items():
    cleaned = clean_section(content)
    knowledge_base[key] = cleaned

In [12]:
knowledge_base['A. Nutrition Assessment']


'[ASPEN_2016_guidelines][A. Nutrition Assessment] Question: Does the use of a nutrition risk indicator identify patients who will most likely benefit from nutrition therapy? \n\nA1. Based on expert consensus, we suggest a determination of nutrition risk (eg, nutritional risk screening [NRS 2002], NUTRIC score) be performed on all patients admitted to the ICU for whom volitional \n\nintake is anticipated to be insufficient. High nutrition risk identifies those patients most likely to benefit from early EN therapy. \n\nRationale: Poor outcomes have been associated with inflammation generated by critical illness that leads to deterioration of nutrition status and malnutrition. However, malnutrition in the critically ill has always been difficult to define. An international consensus group modified definitions to recognize the impact of inflammation. Objective measures of baseline nutrition status have been described by A.S.P.E.N. and the Academy of Nutrition and Dietetics. However, nutrit

In [13]:
knowledge_base['B. Initiate EN']


'[ASPEN_2016_guidelines][B. Initiate EN] Question: What is the benefit of early EN in critically ill adult patients compared with withholding or delaying this therapy? \n\nB1. We recommend that nutrition support therapy in the form of early EN be initiated within 24–48 hours in the critically ill patient who is unable to maintain volitional intake. \n\nRationale: EN supports the functional integrity of the gut by maintaining tight junctions between the intraepithelial cells, stimulating blood flow, and inducing the release of trophic endogenous agents (eg, cholecystokinin, gastrin, bombesin, and bile salts). EN maintains structural integrity by maintaining villous height and supporting the mass of secretory IgAproducing immunocytes (B cells and plasma cells) that compose the gut-associated lymphoid tissue (GALT) and in turn contribute to mucosal-associated lymphoid tissue at distant sites such as the lungs, liver, and kidneys. \n\nAdverse change in gut permeability from loss of functio

In [14]:
# Validation
print("KNOWLEDGE BASE VALIDATION")
print("=" * 50)
print(f"Total documents: {len(knowledge_base)}")

total_words = 0
for doc_name, content in knowledge_base.items():
    word_count = len(content.split())
    total_words += word_count
    status = "Pass" if word_count >= 150 else "Too short!"
    print(f"  {doc_name}: {word_count} words {status}")

print(f"\nTotal words: {total_words}")
print(f"Average words per document: {total_words // len(knowledge_base)}")

if len(knowledge_base) >= 8:
    print("\nDocument count requirement met!")
else:
    print(f"\nYou need at least 8 documents. Currently have: {len(knowledge_base)}")

KNOWLEDGE BASE VALIDATION
Total documents: 22
  A. Nutrition Assessment: 1477 words Pass
  B. Initiate EN: 1707 words Pass
  C. Dosing of EN: 1256 words Pass
  D. Monitoring Tolerance and Adequacy of EN: 2558 words Pass
  E. Selection of Appropriate Enteral Formulation: 1222 words Pass
  F. Adjunctive Therapy: 1334 words Pass
  G. When to Use PN: 1174 words Pass
  H. When Indicated, Maximize Efficacy of PN: 2694 words Pass
  I. Pulmonary Failure: 547 words Pass
  J. Renal Failure: 372 words Pass
  K. Hepatic Failure: 714 words Pass
  L. Acute Pancreatitis: 1741 words Pass
  M. Surgical Subsets: 3 words Too short!
  Trauma: 613 words Pass
  Traumatic Brain Injury: 161 words Pass
  Open Abdomen: 825 words Pass
  Burns: 702 words Pass
  N. Sepsis: 1568 words Pass
  O. Postoperative Major Surgery (SICU Admission Expected): 2272 words Pass
  P. Chronically Critically Ill: 332 words Pass
  Q. Obesity in Critical Illness: 2731 words Pass
  R. Nutrition Therapy End-of-Life Situations: 118 word

---

## Part 4: Document Chunking

Implement a chunking function and processing the knowledge base.

In [15]:
def chunk_document(
    text: str,
    chunk_size: int = 650,
    chunk_overlap: int = 100) -> List[str]:
    """
    Split text into overlapping chunks, respecting sentence boundaries.

    Args:
        text: The text to chunk
        chunk_size: Target size for each chunk (in characters)
        chunk_overlap: Number of characters to overlap between chunks

    Returns:
        List of text chunks
    """
    # Clean the text
    text = text.strip()
    text = re.sub(r'\n+', ' ', text)  # Replace newlines with spaces
    text = re.sub(r'\s+', ' ', text)  # Normalize whitespace

    # Split into sentences (simple approach)
    sentences = re.split(r'(?<=[.!?])\s+', text)

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        # If adding this sentence would exceed chunk size
        if len(current_chunk) + len(sentence) > chunk_size and current_chunk:
            chunks.append(current_chunk.strip())
            # Start new chunk with overlap from previous chunk
            overlap_start = max(0, len(current_chunk) - chunk_overlap)
            current_chunk = current_chunk[overlap_start:] + " " + sentence
        else:
            current_chunk += " " + sentence if current_chunk else sentence

    # Add the last chunk
    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks

print("✅ Chunking function defined")


# Test your chunking function
test_text = list(knowledge_base.values())[1]
test_chunks = chunk_document(test_text)

print("🔍 CHUNKING TEST")
print("=" * 50)
print(f"Original document length: {len(test_text)} characters")
print(f"Number of chunks created: {len(test_chunks)}")
print(f"\nChunk previews:")
print(test_chunks[0][:] + "...." if len(test_chunks) > 0 else "No chunks created")
print("")
print(test_chunks[1][:] + "...." if len(test_chunks) > 0 else "No chunks created")

✅ Chunking function defined
🔍 CHUNKING TEST
Original document length: 10920 characters
Number of chunks created: 24

Chunk previews:
[ASPEN_2016_guidelines][B. Initiate EN] Question: What is the benefit of early EN in critically ill adult patients compared with withholding or delaying this therapy? B1. We recommend that nutrition support therapy in the form of early EN be initiated within 24–48 hours in the critically ill patient who is unable to maintain volitional intake. Rationale: EN supports the functional integrity of the gut by maintaining tight junctions between the intraepithelial cells, stimulating blood flow, and inducing the release of trophic endogenous agents (eg, cholecystokinin, gastrin, bombesin, and bile salts).....

g the release of trophic endogenous agents (eg, cholecystokinin, gastrin, bombesin, and bile salts). EN maintains structural integrity by maintaining villous height and supporting the mass of secretory IgAproducing immunocytes (B cells and plasma cells) t

In [16]:
# Configure chunking parameters
CHUNK_SIZE = 1024
CHUNK_OVERLAP = 128

# Process all documents
all_chunks = []
all_metadatas = []
all_ids = []

for doc_name, document in knowledge_base.items():
  match = re.match(r'(\[.*?\]\[.*?\])', document.strip())
  doc_title = match.group(1) if match else doc_name

  match = re.match(r'(\[.*?\])', document.strip())
  source = match.group(1) if match else doc_name

  chunks = chunk_document(document, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

  for i, chunk in enumerate(chunks):
    contextualized_chunk = chunk if i == 0 else f"{doc_title} {chunk}"

    all_chunks.append(contextualized_chunk)
    all_metadatas.append({
        "source":source,
        "title":doc_title,
        "chunk_index": i
    })
    all_ids.append(f"{doc_title}_chunk_{i}")

# Validation
print("CHUNKING RESULTS")
print("=" * 50)
print(f"Total chunks created: {len(all_chunks)}")
print(f"Chunk size setting: {CHUNK_SIZE}")
print(f"Chunk overlap setting: {CHUNK_OVERLAP}")

CHUNKING RESULTS
Total chunks created: 221
Chunk size setting: 1024
Chunk overlap setting: 128


In [17]:
all_chunks[3], all_metadatas[3], all_ids[3]

('[ASPEN_2016_guidelines][A. Nutrition Assessment] ese scoring systems may increase their applicability in the future by providing guidance as to the role of EN and PN in the ICU. Question: What additional tools, components, or surrogate markers provide useful information when performing nutrition assessments in critically ill adult patients? A2. Based on expert consensus, we suggest that nutrition assessment include an evaluation of comorbid conditions, function of the gastrointestinal (GI) tract, and risk of aspiration. We suggest not using traditional nutrition indicators or surrogate markers, as they are not validated in critical care. Rationale: In the critical care setting, the traditional serum protein markers (albumin, prealbumin, transferrin, retinolbinding protein) are a reflection of the acute-phase response (increases in vascular permeability and reprioritization of hepatic protein synthesis) and do not accurately represent nutrition status in the ICU setting.',
 {'source':

### Chunking Strategy Justification

**TODO:** In the cell below, explain your chunking decisions:

*Double-click to edit this cell*

**Rationale of Chunk Size**

In the ASPEN 2016 guidelines, clincal questions and scenarios are generally 10-20 words (100-200 characters) long. Clincal guideance paragraphs are generally 200-300 words (1500-3000 characters long).

A chunk size of 1200 allows for the entire clinical question to be included in the inital chuck of each document along with crutial clinical context. As we are using medical guidelines, sentences are generally longer and interconnected. Using a chunk size of 1200 and overlap of 170 allows us a more coherent context window will keeping embeddings focused.

**Rationale of Overlap Amount**

We used a larger overlap of 170 to account for the possibility of longer clinical rationales (i.e. considerations of immune modulating formulas in diifferent populatitons). A larger overlap ensures that clinical reasoning between connected chunks is preserved.   

**Additional Modifications**

Originally a 53 page document, we split the paper into seperate documents based on condition type. Citations and tables were also removed. We also elected to use `"abhinand/MedEmbed-small-v0.1"` to better embed medical terminology.



---

## Part 5: Vector Database Setup

Creating a ChromaDB collection and add all chunks with embeddings.

In [18]:
# Initialize ChromaDB client
chroma_client = chromadb.Client()

# Create embedding function
embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="abhinand/MedEmbed-small-v0.1"
)

# embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
#     model_name="abhinand/MedEmbed-base-v0.1"  # upgrade from small
# )

# TODO: Create your collection with a descriptive name
# Include metadata describing your knowledge base

collection = chroma_client.create_collection(
    name="ICU-nutritition-guidelines-db",
    embedding_function=embedding_function,
    metadata={"description": "database of best practice for nutrititon intervention in the adult ICU patient"}  # TODO: Change this
)

print("Collection created!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/297k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/711 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Collection created!


In [19]:
# Add chunks to the collection
collection.add(
    documents=all_chunks,
    metadatas=all_metadatas,
    ids=all_ids
)

print(f"✅ Added {collection.count()} chunks to the database")

✅ Added 221 chunks to the database


In [20]:
# Test retrieval with a sample query
test_query = "Calorie needs for the Acute Respiratory Distress Syndrome (ARDS) patient?"

results = collection.query(
    query_texts=[test_query],
    n_results=6
)

print(f"Test Query: \"{test_query}\"")
print("\nTop 3 Retrieved Chunks:")
for i in range(len(results['documents'][0])):
    print(f"\n{i+1}. Source: {results['metadatas'][0][i].get('source', 'Unknown')}")
    print(f"   Preview: {results['documents'][0][i][:]}...")

Test Query: "Calorie needs for the Acute Respiratory Distress Syndrome (ARDS) patient?"

Top 3 Retrieved Chunks:

1. Source: [ASPEN_2016_guidelines]
   Preview: [ASPEN_2016_guidelines][Trauma] onged stays in the ICU and should undergo timely nutrition reassessment. Energy requirements vary depending on numerous factors. Resting energy expenditure (REE) peaks over 4–5 days but continues to remain high for 9–12 days (with some elevation in energy expenditure persisting for over 21 days). Approximately 16% of total body protein is lost in the first 21 days, with 67% of that protein loss coming from skeletal muscle alone. Energy goals should be in the range of 20–35 kcal/kg/d, depending on the phase of trauma. Lower energy provision is suggested early in the resuscitative phase, with liberalization of energy delivery as the patient enters into the rehabilitation phase. Requirements for protein are similar for other ICU patients but may be at the higher end of the provision range, from 1.2–

In [21]:
results = collection.query(
    query_texts=["Calorie needs for the Acute Respiratory Distress Syndrome (ARDS) patient?"],
    n_results=6  # get more results
)

# print all results with their distances
for i, (doc, distance, metadata) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):
    print(f"{i+1}. Distance: {distance:.4f} | {metadata['title']}")
    print(f"   {doc[:100]}")
    print()

1. Distance: 0.2166 | [ASPEN_2016_guidelines][Trauma]
   [ASPEN_2016_guidelines][Trauma] onged stays in the ICU and should undergo timely nutrition reassessm

2. Distance: 0.2247 | [ASPEN_2016_guidelines][A. Nutrition Assessment]
   [ASPEN_2016_guidelines][A. Nutrition Assessment] e tools to incorporate into nutrition assessment; h

3. Distance: 0.2270 | [ASPEN_2016_guidelines][J. Renal Failure]
   [ASPEN_2016_guidelines][J. Renal Failure] ICU patient, the underlying disease process, preexisting c

4. Distance: 0.2282 | [ASPEN_2016_guidelines][N. Sepsis]
   [ASPEN_2016_guidelines][N. Sepsis] /d) for the initial phase of sepsis, advancing as tolerated after

5. Distance: 0.2289 | [ASPEN_2016_guidelines][Q. Obesity in Critical Illness]
   [ASPEN_2016_guidelines][Q. Obesity in Critical Illness] or all classes of obesity, the goal of the E

6. Distance: 0.2295 | [ASPEN_2016_guidelines][I. Pulmonary Failure]
   [ASPEN_2016_guidelines][I. Pulmonary Failure] Question: What is the optimal carb

---

## Part 6: Building the RAG Pipeline

In [22]:
# Load the extractive question-answering model
# NOTE: this model is a SQuAD extractive QA model -- it must NOT be loaded as "text-generation"
MODEL_NAME = "dmis-lab/biobert-large-cased-v1.1-squad"
device = 0 if torch.cuda.is_available() else -1

try:
    qa_model = pipeline("question-answering", model=MODEL_NAME, tokenizer=MODEL_NAME, device=device)
except (KeyError, ValueError) as e:
    # Fallback for transformers versions without the "question-answering" pipeline task
    print(f"[INFO] QA pipeline unavailable ({e}); using manual extractive QA")
    from transformers import AutoTokenizer, AutoModelForQuestionAnswering
    _tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    _mdl = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME).to("cuda" if device == 0 else "cpu").eval()

    def qa_model(question: str, context: str, max_length: int = 384, stride: int = 128, max_answer_len: int = 60):
        enc = _tok(question, context, truncation="only_second", max_length=max_length, stride=stride,
                   return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length",
                   return_tensors="pt")
        offsets = enc.pop("offset_mapping"); enc.pop("overflow_to_sample_mapping", None)
        enc = {k: v.to(_mdl.device) for k, v in enc.items()}
        with torch.no_grad():
            out = _mdl(**enc)
        best = {"answer": "", "score": 0.0}
        for w in range(out.start_logits.shape[0]):
            seq_ids = _tok(question, context, truncation="only_second", max_length=max_length, stride=stride,
                           return_overflowing_tokens=True).sequence_ids(w)
            ctx_mask = torch.tensor([s == 1 for s in seq_ids] + [False] * (max_length - len(seq_ids)), device=_mdl.device)
            s = out.start_logits[w].masked_fill(~ctx_mask, -1e4).softmax(-1)
            e = out.end_logits[w].masked_fill(~ctx_mask, -1e4).softmax(-1)
            scores = torch.triu(s[:, None] * e[None, :]) - torch.triu(torch.ones_like(s[:, None] * e[None, :]), diagonal=max_answer_len)
            idx = scores.argmax(); i, j = divmod(idx.item(), scores.shape[1]); sc = scores[i, j].item()
            if sc > best["score"]:
                best = {"answer": context[offsets[w][i][0]:offsets[w][j][1]], "score": sc}
        return best

print("QA model loaded!")


config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

[INFO] QA pipeline unavailable ('The task question-answering has been removed from transformers. Use the `text-generation` pipeline with a chat model instead. See the migration guide for more details: https://github.com/huggingface/transformers/blob/main/MIGRATION_GUIDE_V5.md#pipelines'); using manual extractive QA


vocab.txt:   0%|          | 0.00/467k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.45GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.45GB            

model.safetensors: downloading bytes:           |  0.00B            

QA model loaded!


In [23]:
# ============================================================
# TODO: Implement your RAG Pipeline class
# ============================================================

class RAGAssistant:
    """
    A complete RAG-powered knowledge assistant.

    Your implementation must include:
    1. Document retrieval from ChromaDB
    2. Answer generation using the QA model
    3. Confidence-based response handling
    4. Source attribution
    """

    def __init__(self, collection, qa_model, n_results: int = 8, confidence_threshold: float = 0.3):
        """
        Initialize the RAG assistant.

        Args:
            collection: ChromaDB collection with embedded documents
            qa_model: Hugging Face QA pipeline
            confidence_threshold: Minimum confidence to provide a direct answer
        """
        self.collection = collection
        self.qa_model = qa_model
        self.confidence_threshold = confidence_threshold
        self.n_results = n_results

    def retrieve(self, query: str, n_results: int = 8) -> Dict:
        """
        Retrieve relevant documents for a query.

        Args:
            query: The user's question
            n_results: Number of documents to retrieve

        Returns:
            Dict containing documents, metadatas, and ids
        """
        # TODO: Implement retrieval
        results = self.collection.query(
            query_texts=[query],
            n_results = self.n_results
        )

        return results

    def generate_answer(self, question: str, context: str) -> Dict:
        """
        Generate an answer from the given context.

        Args:
            question: The user's question
            context: Retrieved context to answer from

        Returns:
            Dict with answer and confidence score
        """
        # TODO: Implement answer generation

        result = self.qa_model(question=question,
                               context=context)

        return {
            'answer':result['answer'],
            'confidence':result['score']
        }

    def ask(self, question: str, n_results:int = 8) -> Dict:
        """
        Complete RAG pipeline: retrieve and generate answer.

        This method should:
        1. Retrieve relevant documents
        2. Combine them into context
        3. Generate an answer
        4. Handle low confidence appropriately
        5. Include source attribution

        Args:
            question: The user's question
            n_results: Number of documents to retrieve

        Returns:
            Dict with:
            - question: Original question
            - answer: Generated answer (or fallback message)
            - confidence: Confidence score
            - sources: List of source documents used
            - is_confident: Boolean indicating if confidence met threshold
        """
        # TODO: Implement the complete pipeline

        retrieved = self.retrieve(query=question,
                                  n_results = self.n_results)

        #context = " ".join(retrieved['documents'][0])

        top_chunks = retrieved['documents'][0][:2] #choose top 5 articles
        context = "\n\n".join(top_chunks)

        answer_result = self.generate_answer(question=question,
                                             context=context)

        sources = [meta['title'] for meta in retrieved['metadatas'][0]]
        # print({
        #     'question': question,
        #     'answer': answer_result['answer'],
        #     'confidence': answer_result['confidence'],
        #     'sources': sources,
        #     'context': context
        # })
        return {
            'question': question,
            'answer': answer_result['answer'],
            'confidence': answer_result['confidence'],
            'is_confident': answer_result['confidence'] > self.confidence_threshold,
            'sources': sources,
            'context': context
        }

    def format_response(self, result: Dict) -> str:
        """
        Format the result into a user-friendly response string.

        Args:
            result: Output from the ask() method

        Returns:
            Formatted string response
        """
        # TODO: Implement response formatting

        question = result['question']

        #print(f"Question: {question}")
        #print("="*50)

        #print(f"\n" + "="*50)
        print(f"\n✅ Answer: {result['answer']}")
        print(f"   Confidence: {result['confidence']:.2%}")
        print(f"   Sources: {', '.join(set(result['sources']))}")
        return ""


# Create your assistant
assistant = RAGAssistant(collection, qa_model, n_results=30, confidence_threshold=0.5)

print("✅ RAG Assistant created!")

✅ RAG Assistant created!


In [29]:
test_questions = [
    "What are the calorie needs for a patient with Acute Respiratory Distress Syndrome (ARDS)?",
    "Is the routine use of disease specific formulas (i.e. diabetic) indicated in critical care?",
    "When should PN we initiated for a patient in unable to tolerate tube feeds (i.e. bowel discontinuity)?",
    'What is the standard calorie provision for the ICU patient?'
]

print("INITIAL TESTING")
print("=" * 60)
for question in test_questions:
    result = assistant.ask(question)
    print(f"\nQ: {question}")
    print(assistant.format_response(result))

INITIAL TESTING

Q: What are the calorie needs for a patient with Acute Respiratory Distress Syndrome (ARDS)?

✅ Answer: 20–35 kcal/kg/d
   Confidence: 51.56%
   Sources: [ASPEN_2016_guidelines][N. Sepsis], [ASPEN_2016_guidelines][H. When Indicated, Maximize Efficacy of PN], [ASPEN_2016_guidelines][I. Pulmonary Failure], [ASPEN_2016_guidelines][J. Renal Failure], [ASPEN_2016_guidelines][C. Dosing of EN], [ASPEN_2016_guidelines][Open Abdomen], [ASPEN_2016_guidelines][Burns], [ASPEN_2016_guidelines][K. Hepatic Failure], [ASPEN_2016_guidelines][A. Nutrition Assessment], [ASPEN_2016_guidelines][Q. Obesity in Critical Illness], [ASPEN_2016_guidelines][Trauma]


Q: Is the routine use of disease specific formulas (i.e. diabetic) indicated in critical care?

✅ Answer: Disease-specific and severe fluid-restricted formulas may be rarely used in a small percentage of patients on a case-by-case basis
   Confidence: 8.35%
   Sources: [ASPEN_2016_guidelines][H. When Indicated, Maximize Efficacy of P

---

## Part 7: Comprehensive Testing

Test your system thoroughly with at least 15 questions covering different types and difficulty levels.

In [25]:
# ============================================================
# TODO: Create your comprehensive test suite
# ============================================================

# Organize your test questions by category
test_suite = {
    "simple_facts": [
        # Questions with straightforward, single-fact answers
        # Example: "What is the price of X?" "When does Y open?"
        "What is the standard calorie range for the ICU patient?",
        "What is the standard protein range for the ICU patient?",
        "What is the maximum safe protein provision for the ICU patient?",
        "What is the best method for determining energy needs in the critically ill adult patient?",
        "Are the bowel sounds required before the initation of enteral tube feeds?",
    ],

    "detailed_explanations": [
        # Questions requiring more detailed answers
        # Example: "How does X work?" "What are the features of Y?"
        "What considerations should be made for enteral feeds when a patient is on pressors?",
        "What are the contraindications against the initation of parenteral nutrition?",
        "How do you mitigate risk of fungemia in parenteral nutrition?",
        "What is the advantage of using standardized commercially available PN (premixed PN) versus compounded PN admixtures?",
        "Explain the progression of acute phase sepsis and shock in the ICU patient.",
    ],

    "comparison_or_complex": [
        # More complex questions
        # Example: "What's the difference between X and Y?"
        "What are the protein requirements for a burn patient versus that of a standard ICU patient?",
        "What is the optimal timing for initiating supplemental PN when EN does not meet energy or protein goals in the patient at high or low nutrition risk?",
    ],

    "edge_cases": [
        # Questions NOT answerable from your knowledge base
        # These test your fallback handling
        "What are the calorie needs a patient with both severe burns and is morbidly obese?",
        "When should supplemental PN be discontinued?",
        "When can intravenous lipid emulsions (IVFE) be safely started for a patient with fungemia?",
    ]
}



In [26]:
# ============================================================
# Run comprehensive tests and collect results
# ============================================================

test_results = []

print("COMPREHENSIVE TEST RESULTS")
print("=" * 70)

for category, questions in test_suite.items():
    print(f"\n\n{'='*70}")
    print(f"CATEGORY: {category.upper().replace('_', ' ')}")
    print("=" * 70)

    for question in questions:
        result = assistant.ask(question)

        # Store result for analysis
        test_results.append({
            "category": category,
            "question": question,
            "answer": result["answer"],
            "confidence": result["confidence"],
            "is_confident": result["is_confident"],
            "sources": result["sources"]
        })

        # Display result
        confidence_indicator = "✅" if result["is_confident"] else "⚠️"
        print(f"\n{confidence_indicator} Q: {question}")
        print(f"   A: {result['answer']}")
        print(f"   Confidence: {result['confidence']:.2%} | Sources: {', '.join(result['sources'][:2])}")

COMPREHENSIVE TEST RESULTS


CATEGORY: SIMPLE FACTS

⚠️ Q: What is the standard calorie range for the ICU patient?
   A: 25 kcal/kg/d
   Confidence: 12.68% | Sources: [ASPEN_2016_guidelines][Q. Obesity in Critical Illness], [ASPEN_2016_guidelines][A. Nutrition Assessment]

✅ Q: What is the standard protein range for the ICU patient?
   A: 1.2–2.0 g/kg actual body weight per day
   Confidence: 76.79% | Sources: [ASPEN_2016_guidelines][C. Dosing of EN], [ASPEN_2016_guidelines][C. Dosing of EN]

⚠️ Q: What is the maximum safe protein provision for the ICU patient?
   A: 1.2–2.0 g/kg actual body weight per day
   Confidence: 21.83% | Sources: [ASPEN_2016_guidelines][C. Dosing of EN], [ASPEN_2016_guidelines][C. Dosing of EN]

⚠️ Q: What is the best method for determining energy needs in the critically ill adult patient?
   A: indirect calorimetry
   Confidence: 47.77% | Sources: [ASPEN_2016_guidelines][A. Nutrition Assessment], [ASPEN_2016_guidelines][Burns]

⚠️ Q: Are the bowel sounds requ

In [27]:
# ============================================================
# Analyze test results
# ============================================================

print("\n" + "=" * 70)
print("TEST RESULTS SUMMARY")
print("=" * 70)

total_questions = len(test_results)
confident_answers = sum(1 for r in test_results if r["is_confident"])
avg_confidence = sum(r["confidence"] for r in test_results) / total_questions

print(f"\nTotal questions tested: {total_questions}")
print(f"Confident answers: {confident_answers} ({confident_answers/total_questions:.1%})")
print(f"Low confidence answers: {total_questions - confident_answers}")
print(f"Average confidence: {avg_confidence:.2%}")

# By category
print("\nResults by category:")
for category in test_suite.keys():
    category_results = [r for r in test_results if r["category"] == category]
    cat_confident = sum(1 for r in category_results if r["is_confident"])
    cat_avg = sum(r["confidence"] for r in category_results) / len(category_results)
    print(f"  {category}: {cat_confident}/{len(category_results)} confident, avg confidence: {cat_avg:.2%}")


TEST RESULTS SUMMARY

Total questions tested: 15
Confident answers: 2 (13.3%)
Low confidence answers: 13
Average confidence: 30.42%

Results by category:
  simple_facts: 1/5 confident, avg confidence: 38.68%
  detailed_explanations: 1/5 confident, avg confidence: 35.23%
  comparison_or_complex: 0/2 confident, avg confidence: 20.92%
  edge_cases: 0/3 confident, avg confidence: 14.98%


---

## Part 8: Analysis

### 8.1 Performance Analysis

Our RAG system was able to handle simple facts (i.e. What is the standard protein range for the ICU patient? -> 1.2 g to 2.0 g/kg). General medical guidance and straight figures can be found in sigular documents and in clear, direct sentences.  

Our system still struggles with cases that deviated from baseline patients (i.e. bowel discontinuity or obesity). This becuase our system was not able to recognize that questions were specifically looking for cases that deviated from "normal" patients. Logic on clincal guidlines for these cases also spanned multiple documents.

### 8.3 Future Improvements

-----

In document preparation we used an open source libary `pymupdf4llm` to first convert a PDF into markdown. We then relied on text/characters at land marks to split the research paper into different documents. For a cleaner knowledge base we can manually convert the PDF into plain text, allowing us tighter control over how the knowledge base is structured. More intentional hierarchical chunking can also be done to make sure the scope/context of each chunk is limited to specific clinical scenarios.

We can use query expasion/generation so that our queries more closely match our embeddings nad vectors. Likewise, for questions that require synthesis of data that spread across multiple documnets was can use generative models such as`google/flan-t5-base` to provide more comprehenisve answers. At 250 MB, this model would still be relatively light weight and runnable on local CPUs.


---

## Part 9: Demo Showcase


In [28]:
def run_demo(assistant, demo_questions: List[str]):
    """
    Run a polished demo of the RAG assistant.
    """
    print("\n" + "="*70)
    print(f"{PROJECT_INFO['topic']} - Knowledge Assistant Demo")
    print(f"   {PROJECT_INFO['description']}")
    print("="*70)

    for i, question in enumerate(demo_questions, 1):
        print(f"\n{'─'*70}")
        print(f"User Question {i}:")
        print(f"   \"{question}\"")
        print()

        result = assistant.ask(question)

        print(f"Assistant Response:")
        print(f"   {result['answer']}")
        print()
        print(f"   Sources: {', '.join(result['sources'][:2])}")
        print(f"   Confidence: {result['confidence']:.1%}")

    print(f"\n{'='*70}")
    print("Demo complete!")
    print("="*70)


# Select 5 of your best questions for the demo
demo_questions = [
    # TODO: Choose 5 questions that showcase your system well
    "What is the standard protein range for the ICU patient?",
    "What is the gold-standard for determining energy needs in the ICU?",
    "Are the bowel sounds required before the initation of enteral tube feeds",
    "Explain the progression of the acute phase repsonse and shock in the ICU patient?",
    "When should parenteral nutrition be initatied?",
]

run_demo(assistant, demo_questions)


Nutrition Intervention in the ICU - Knowledge Assistant Demo
   This RAG LLM is trained with ASPENs 2016 and 2021 practice guidelines for nutrition intervention in the ICU.

──────────────────────────────────────────────────────────────────────
User Question 1:
   "What is the standard protein range for the ICU patient?"

Assistant Response:
   1.2–2.0 g/kg actual body weight per day

   Sources: [ASPEN_2016_guidelines][C. Dosing of EN], [ASPEN_2016_guidelines][C. Dosing of EN]
   Confidence: 76.8%

──────────────────────────────────────────────────────────────────────
User Question 2:
   "What is the gold-standard for determining energy needs in the ICU?"

Assistant Response:
   indirect calorimetry

   Sources: [ASPEN_2016_guidelines][A. Nutrition Assessment], [ASPEN_2016_guidelines][A. Nutrition Assessment]
   Confidence: 80.3%

──────────────────────────────────────────────────────────────────────
User Question 3:
   "Are the bowel sounds required before the initation of enteral t